### Generate RAG-ready chunks

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.silver.policy_chunks_raw AS
WITH prepped AS (
  SELECT
    file_name,
    policy_id,
    ai_prep_search(parsed) AS result
  FROM policyiq.bronze.policy_documents_parsed
)
SELECT
  file_name,
  policy_id,
  chunk.value:chunk_id::STRING        AS chunk_id,
  chunk.value:chunk_position::INT     AS chunk_position,
  chunk.value:chunk_to_retrieve::STRING AS chunk_text,
  chunk.value:chunk_to_embed::STRING  AS chunk_embed_text
FROM prepped, LATERAL variant_explode(result:document.contents) AS chunk;

In [0]:
%sql
select * from policyiq.silver.policy_chunks_raw

### Enrich with registry metadata and a stable primary key

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.silver.policy_chunks AS
SELECT
  concat(c.policy_id, '_', c.chunk_position)   AS policy_chunk_id,
  c.policy_id,
  r.policy_name,
  r.policy_domain,
  r.issuing_authority,
  r.version,
  c.file_name,
  c.chunk_position,
  c.chunk_text,
  c.chunk_embed_text,
  length(c.chunk_text)                          AS chunk_char_length,
  current_timestamp()                           AS ingested_at
FROM policyiq.silver.policy_chunks_raw c
JOIN policyiq.bronze.policy_registry r
  ON c.policy_id = r.policy_id;

In [0]:
%sql
select * from policyiq.silver.policy_chunks

In [0]:
%sql
SELECT policy_id, count(*) AS chunk_count,
       round(avg(chunk_char_length)) AS avg_chunk_len,
       min(chunk_char_length) AS min_len,
       max(chunk_char_length) AS max_len
FROM policyiq.silver.policy_chunks
GROUP BY policy_id
ORDER BY policy_id;

In [0]:
%sql
SELECT policy_id, chunk_position, chunk_text
FROM policyiq.silver.policy_chunks
WHERE policy_id = 'HR_LEAVE_2026'
ORDER BY chunk_position
LIMIT 5;

In [0]:
%sql
SELECT policy_id, chunk_position, chunk_char_length, chunk_text
FROM policyiq.silver.policy_chunks
WHERE policy_id = 'CYBERSEC_2026'
ORDER BY chunk_char_length ASC
LIMIT 5;

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.silver.policy_chunks AS
SELECT * FROM policyiq.silver.policy_chunks
WHERE chunk_char_length >= 30;

SELECT policy_id, count(*) AS chunk_count, min(chunk_char_length) AS min_len
FROM policyiq.silver.policy_chunks
GROUP BY policy_id
ORDER BY policy_id;

### Build the KPI-side Silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.silver.kpi_compliance_facts AS
WITH joined AS (
  SELECT ka.branch_id, b.branch_name, b.division, b.region_risk_classification,
    ka.policy_domain, ka.kpi_name, kr.kpi_display_name, kr.policy_id, kr.policy_section_ref,
    CAST(ka.actual_value AS DOUBLE) AS actual_value, CAST(kr.threshold_value AS DOUBLE) AS threshold_value,
    kr.threshold_operator, kr.unit, kr.direction, ka.reporting_period, CAST(ka.as_of_date AS DATE) AS as_of_date,
    CASE
      WHEN kr.threshold_operator = '>=' THEN CAST(ka.actual_value AS DOUBLE) >= CAST(kr.threshold_value AS DOUBLE)
      WHEN kr.threshold_operator = '<=' THEN CAST(ka.actual_value AS DOUBLE) <= CAST(kr.threshold_value AS DOUBLE)
      WHEN kr.threshold_operator = '='  THEN CAST(ka.actual_value AS DOUBLE) =  CAST(kr.threshold_value AS DOUBLE)
    END AS is_compliant,
    ROUND(CAST(ka.actual_value AS DOUBLE) - CAST(kr.threshold_value AS DOUBLE), 2) AS raw_gap
  FROM policyiq.bronze.kpi_actuals ka
  JOIN policyiq.bronze.kpi_registry kr ON ka.kpi_name = kr.kpi_name
  JOIN policyiq.bronze.branch_dim   b  ON ka.branch_id = b.branch_id
)
SELECT *, CASE WHEN is_compliant THEN 'Compliant' ELSE 'Non-Compliant' END AS compliance_status
FROM joined;

SELECT compliance_status, count(*) FROM policyiq.silver.kpi_compliance_facts GROUP BY compliance_status;

In [0]:
%sql
SELECT compliance_status, count(*) AS n
FROM policyiq.silver.kpi_compliance_facts
GROUP BY compliance_status;

In [0]:
%sql
SELECT branch_id, branch_name, count(*) AS violation_count
FROM policyiq.silver.kpi_compliance_facts
WHERE is_compliant = false
GROUP BY branch_id, branch_name
ORDER BY violation_count DESC;